In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0124.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0415.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0421.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0220.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0300.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0215.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0045.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0097.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0212.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0032.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0445.png
/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai/sdxl/sdxl_0055.png
/kaggle/input/datasets/arunmass/ai-image

In [2]:
# ==========================================
# AI IMAGE DETECTOR - OPTIMIZED FOR 2x T4 GPUs
# Resolution: 512x512 (Maximum forensic detail)
# ==========================================

import os
import sys
import torch
import gc
import numpy as np
from typing import Dict, Any
from io import BytesIO
from PIL import Image
import random

In [4]:
# ==========================================
# MULTI-GPU SETUP
# ==========================================
print("🔍 Detecting hardware...")
if torch.cuda.is_available():
    NUM_GPUS = torch.cuda.device_count()
    print(f"🎮 {NUM_GPUS} GPU(s) Detected!")
    for i in range(NUM_GPUS):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"   Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
    torch.cuda.empty_cache()
    gc.collect()
    DEVICE = "cuda"
else:
    print("❌ ERROR: No GPU detected!")
    sys.exit(1)

🔍 Detecting hardware...
🎮 2 GPU(s) Detected!
   GPU 0: Tesla T4
   Memory: 14.6 GB
   GPU 1: Tesla T4
   Memory: 14.6 GB


In [5]:
# ==========================================
# IMPORTS
# ==========================================
from datasets import Dataset, Image as HFImage, ClassLabel
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from torchvision.transforms import (
    Compose, Resize, ToTensor, Normalize,
    RandomHorizontalFlip, RandomRotation, RandomResizedCrop
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F

2026-02-10 10:15:07.315362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770718507.502812      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770718507.558594      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770718508.024915      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770718508.024949      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770718508.024952      55 computation_placer.cc:177] computation placer alr

In [6]:
# ==========================================
# CONFIGURATION - OPTIMIZED FOR 2x T4 @ 512x512
# ==========================================

# === PATHS (Update these for Kaggle) ===
AI_FOLDER = "/kaggle/input/datasets/arunmass/ai-image-vs-real-image/ai/ai"  # UPDATE THIS
REAL_FOLDER = "/kaggle/input/datasets/arunmass/ai-image-vs-real-image/real/real"  # UPDATE THIS
OUTPUT_DIR = "/kaggle/working/zero_ai_forensic_512"

# === MODEL SELECTION ===
MODEL_ID = "Ateeqq/ai-vs-human-image-detector"

In [7]:
# === TRAINING CONFIG - OPTIMIZED FOR 2x T4 ===
INPUT_RESOLUTION = 512  # 🔥 HIGH RESOLUTION for maximum forensic detail
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5

# Batch size per GPU (2 T4s with 16GB each can handle this)
BATCH_SIZE_PER_GPU = 10  # 10 per GPU × 2 GPUs = 20 total
GRAD_ACCUM_STEPS = 4     # Effective batch = 20 × 4 = 80
LEARNING_RATE = 2e-6     # Lower LR for high resolution
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
DROPOUT_RATE = 0.2

# Multi-GPU training strategy
USE_DATA_PARALLEL = True  # Set to False for DistributedDataParallel if needed

print("="*70)
print("⚙️ OPTIMIZED FOR 2x T4 @ 512x512")
print("="*70)
print(f"Resolution:      {INPUT_RESOLUTION}x{INPUT_RESOLUTION} (HIGH)")
print(f"GPUs:            {NUM_GPUS}")
print(f"Batch/GPU:       {BATCH_SIZE_PER_GPU}")
print(f"Total Batch:     {BATCH_SIZE_PER_GPU * NUM_GPUS}")
print(f"Gradient Accum:  {GRAD_ACCUM_STEPS}")
print(f"Effective BS:    {BATCH_SIZE_PER_GPU * NUM_GPUS * GRAD_ACCUM_STEPS}")
print(f"Learning Rate:   {LEARNING_RATE}")
print(f"Model:           {MODEL_ID}")
print("="*70)

⚙️ OPTIMIZED FOR 2x T4 @ 512x512
Resolution:      512x512 (HIGH)
GPUs:            2
Batch/GPU:       10
Total Batch:     20
Gradient Accum:  4
Effective BS:    80
Learning Rate:   2e-06
Model:           Ateeqq/ai-vs-human-image-detector


In [8]:
# ==========================================
# FORENSIC-AWARE AUGMENTATIONS
# ==========================================

class RandomJPEGCompression:
    """Apply random JPEG compression to simulate real-world scenarios"""
    def __init__(self, quality_range=(70, 98), p=0.5):
        self.quality_range = quality_range
        self.p = p
    
    def __call__(self, img):
        if random.random() < self.p:
            quality = random.randint(*self.quality_range)
            buffer = BytesIO()
            if img.mode != 'RGB':
                img = img.convert('RGB')
            img.save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).copy()
            buffer.close()
        return img


class QualityBalancer:
    """Add subtle noise to prevent quality-based classification"""
    def __init__(self, p=0.3):
        self.p = p
    
    def __call__(self, img):
        if random.random() < self.p:
            img_array = np.array(img)
            # Very subtle Gaussian noise
            noise = np.random.normal(0, 1.5, img_array.shape)
            img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_array)
        return img


class AdaptiveResize:
    """Smart resize that preserves aspect ratio when possible"""
    def __init__(self, size=512):
        self.size = size
    
    def __call__(self, img):
        w, h = img.size
        # If already close to target size, just resize
        if abs(w - self.size) < 50 and abs(h - self.size) < 50:
            return img.resize((self.size, self.size), Image.BICUBIC)
        
        # If very large, downsample in steps to preserve detail
        max_dim = max(w, h)
        if max_dim > self.size * 2:
            # First resize to 2x target
            scale = (self.size * 2) / max_dim
            new_w, new_h = int(w * scale), int(h * scale)
            img = img.resize((new_w, new_h), Image.LANCZOS)
        
        # Final resize to target
        return img.resize((self.size, self.size), Image.BICUBIC)


In [9]:
# ==========================================
# 1. LOAD DATASET
# ==========================================
print(f"\n📂 Loading dataset...")
image_paths = []
labels = []
sources = []
classes = ['ai', 'hum']
class_map = {'ai': 0, 'hum': 1}

valid_extensions = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff'}

def load_images_recursive(folder_path, label):
    """Load images and track subfolder sources"""
    count = 0
    for root, dirs, files in os.walk(folder_path):
        subfolder = os.path.basename(root)
        for file in files:
            if os.path.splitext(file)[1].lower() in valid_extensions:
                image_paths.append(os.path.join(root, file))
                labels.append(label)
                sources.append(f"{classes[label]}_{subfolder}")
                count += 1
    return count

if not os.path.exists(AI_FOLDER):
    raise ValueError(f"❌ AI folder not found: {AI_FOLDER}")
ai_count = load_images_recursive(AI_FOLDER, class_map['ai'])
print(f"   Found {ai_count:,} AI images")

if not os.path.exists(REAL_FOLDER):
    raise ValueError(f"❌ Real folder not found: {REAL_FOLDER}")
real_count = load_images_recursive(REAL_FOLDER, class_map['hum'])
print(f"   Found {real_count:,} Real images")

# Calculate class weights
total_samples = ai_count + real_count
weight_ai = total_samples / (2 * ai_count)
weight_real = total_samples / (2 * real_count)
class_weights = torch.tensor([weight_ai, weight_real]).to(DEVICE)

print(f"\n📊 Class Distribution:")
print(f"   AI:   {ai_count:,} ({ai_count/total_samples*100:.1f}%)")
print(f"   Real: {real_count:,} ({real_count/total_samples*100:.1f}%)")
print(f"   Class weights: AI={weight_ai:.3f}, Real={weight_real:.3f}")

# Create dataset
dataset = Dataset.from_dict({
    "image": image_paths,
    "label": labels,
    "source": sources
}).cast_column("image", HFImage())

features = dataset.features.copy()
features["label"] = ClassLabel(names=classes)
dataset = dataset.cast(features)

# Stratified split
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    range(len(dataset)),
    test_size=0.15,
    stratify=dataset['label'],
    random_state=42
)

train_ds = dataset.select(train_idx)
test_ds = dataset.select(test_idx)

print(f"✅ Train: {len(train_ds):,} | Test: {len(test_ds):,}\n")


📂 Loading dataset...
   Found 3,099 AI images
   Found 5,321 Real images

📊 Class Distribution:
   AI:   3,099 (36.8%)
   Real: 5,321 (63.2%)
   Class weights: AI=1.359, Real=0.791


Casting the dataset:   0%|          | 0/8420 [00:00<?, ? examples/s]

✅ Train: 7,157 | Test: 1,263



In [34]:
# ==========================================
# 2. FORENSIC PREPROCESSING @ 512x512
# ==========================================
print("🖼️ Setting up preprocessing for 512x512...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

# TRAINING TRANSFORMS - FORENSIC OPTIMIZED
train_transforms = Compose([
    # JPEG compression (simulate real-world)
    RandomJPEGCompression(quality_range=(70, 98), p=0.4),
    
    # Quality balancing
    QualityBalancer(p=0.25),
    
    # AGGRESSIVE CROPPING - Focus on texture
    # At 512x512, we can afford more aggressive crops
    RandomResizedCrop(
        INPUT_RESOLUTION,
        scale=(0.3, 1.0),  # 30%-100% crops (more aggressive at high res)
        ratio=(0.95, 1.05),
        interpolation=Image.BICUBIC,
        antialias=True
    ),
    
    # Minimal geometric augmentation
    RandomHorizontalFlip(p=0.5),
    RandomRotation(degrees=3),  # Very subtle
    
    # NO ColorJitter - destroys forensic evidence!
    # NO GaussianBlur - destroys noise patterns!
    
    ToTensor(),
    normalize,
])

# VALIDATION TRANSFORMS - NO AUGMENTATION
val_transforms = Compose([
    AdaptiveResize(INPUT_RESOLUTION),
    ToTensor(),
    normalize,
])

def train_transform_fn(examples):
    # 1. Create the pixel values
    examples["pixel_values"] = [
        train_transforms(img.convert("RGB")) for img in examples["image"]
    ]
    
    # 2. DELETE the raw 'image' column so the collator doesn't choke on it
    #    (This fixes the RuntimeError)
    del examples["image"]
    
    # 3. Clean up other unused columns to be safe
    if "source" in examples:
        del examples["source"]
        
    return examples

def val_transform_fn(examples):
    examples["pixel_values"] = [
        val_transforms(img.convert("RGB")) for img in examples["image"]
    ]
    # Delete raw image here too
    del examples["image"]
    if "source" in examples:
        del examples["source"]
    return examples

print("🔄 Applying transforms (Updated to clean raw images)...")
train_ds = train_ds.with_transform(train_transform_fn)
test_ds = test_ds.with_transform(val_transform_fn)

🖼️ Setting up preprocessing for 512x512...
🔄 Applying transforms (Updated to clean raw images)...


In [35]:
# ==========================================
# 3. CUSTOM TRAINER WITH CLASS WEIGHTS
# ==========================================

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Weighted cross-entropy
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights,
            label_smoothing=LABEL_SMOOTHING
        )
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [36]:
# ==========================================
# 4. METRICS WITH DETAILED BREAKDOWN
# ==========================================

def compute_metrics(eval_pred) -> Dict[str, float]:
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    
    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    
    # Per-class metrics
    ai_accuracy = tn / (tn + fp) if (tn + fp) > 0 else 0
    real_accuracy = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # False positive/negative rates
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "ai_accuracy": ai_accuracy,
        "real_accuracy": real_accuracy,
        "fpr": fpr,  # False positive rate
        "fnr": fnr,  # False negative rate
    }

In [40]:
# ==========================================
# 5. MODEL SETUP WITH ARTIFACT ATTENTION HEAD & RESIZING
# ==========================================

print(f"\n🤖 Loading model: {MODEL_ID}...")

# ---------------------------------------------------------
# A. DEFINE CUSTOM LAYERS
# ---------------------------------------------------------
class ArtifactAttention(nn.Module):
    """
    Feature-wise attention to re-calibrate importance of forensic artifacts.
    Mathematically similar to Squeeze-and-Excitation (SE) blocks.
    """
    def __init__(self, hidden_dim=512):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, hidden_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # x shape: [batch_size, hidden_dim]
        weights = self.attention(x)
        return x * weights  # Scale features by their "forensic importance"


class ForensicClassifier(nn.Module):
    """Custom forensic classification head with attention"""
    def __init__(self, in_features, num_classes=2):
        super().__init__()
        
        # 1. Feature Expansion (Project to high-dim space)
        self.fc1 = nn.Linear(in_features, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.dropout1 = nn.Dropout(p=0.4)
        
        # 2. The Innovation: Artifact Attention Mechanism
        self.attention = ArtifactAttention(hidden_dim=1024)
        
        # 3. Feature Compression
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(p=0.3)
        
        # 4. Final Classification
        self.fc3 = nn.Linear(512, num_classes)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.gelu(x)
        x = self.dropout1(x)
        
        # Apply Attention (The "Brain")
        x = self.attention(x)
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = F.gelu(x)
        x = self.dropout2(x)
        
        # Output
        x = self.fc3(x)
        return x

# ---------------------------------------------------------
# B. DEFINE RESIZING LOGIC (THE FIX)
# ---------------------------------------------------------
def resize_siglip_embeddings(model, new_resolution):
    """
    Resizes the SigLIP/ViT position embeddings from 224x224 (pretrained)
    to support the custom 512x512 forensic resolution.
    """
    print(f"\n🔧 Checking model position embeddings for {new_resolution}x{new_resolution}...")
    
    # 1. Locate the vision backbone
    vision_model = None
    if hasattr(model, "vision_model"):
        vision_model = model.vision_model
    elif hasattr(model, "siglip"):
        vision_model = model.siglip.vision_model
    elif hasattr(model, "vit"):
        vision_model = model.vit
        
    if vision_model is None:
        print("⚠️ Could not locate vision backbone. Skipping resize.")
        return

    # 2. Get Embedding Details
    embeddings = vision_model.embeddings
    patch_size = embeddings.patch_size
    
    # Calculate required patches
    new_num_patches = (new_resolution // patch_size) ** 2
    old_num_patches = embeddings.num_patches
    
    if old_num_patches == new_num_patches:
        print("✅ Embeddings already match target resolution.")
        return

    print(f"   Resizing embeddings: {old_num_patches} patches -> {new_num_patches} patches")

    # 3. Extract and Reshape Old Embeddings
    # Shape: [num_patches, dim]
    old_pos_emb = embeddings.position_embedding.weight.data
    embed_dim = old_pos_emb.shape[-1]
    
    # Calculate grid size (e.g., 14x14 for 224px, 32x32 for 512px)
    old_grid_size = int(old_num_patches**0.5)
    new_grid_size = int(new_num_patches**0.5)
    
    # Reshape to [1, dim, grid, grid] for interpolation
    # (196, 768) -> (1, 14, 14, 768) -> (1, 768, 14, 14)
    old_pos_emb = old_pos_emb.reshape(1, old_grid_size, old_grid_size, embed_dim).permute(0, 3, 1, 2)
    
    # 4. Interpolate (Bicubic Resize)
    new_pos_emb = F.interpolate(
        old_pos_emb, 
        size=(new_grid_size, new_grid_size), 
        mode='bicubic', 
        align_corners=False
    )
    
    # 5. Flatten back to [new_num_patches, dim]
    new_pos_emb = new_pos_emb.permute(0, 2, 3, 1).reshape(new_num_patches, embed_dim)
    
    # 6. Apply to Model
    new_embedding_layer = nn.Embedding(new_num_patches, embed_dim)
    new_embedding_layer.weight.data = new_pos_emb.to(model.device)
    embeddings.position_embedding = new_embedding_layer
    
    # Update Config/Metadata
    embeddings.num_patches = new_num_patches
    embeddings.image_size = new_resolution
    embeddings.register_buffer("position_ids", torch.arange(new_num_patches).expand((1, -1)))
    model.config.vision_config.image_size = new_resolution
    
    print(f"✅ Successfully resized position embeddings to {new_resolution}x{new_resolution}")


# ---------------------------------------------------------
# C. LOAD AND CONFIGURE MODEL
# ---------------------------------------------------------
# Load base model
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(classes),
    id2label={0: "ai", 1: "human"},
    label2id={"ai": 0, "human": 1},
    ignore_mismatched_sizes=True,
)

# === APPLY THE FIX ===
resize_siglip_embeddings(model, INPUT_RESOLUTION)

# === ATTACH CUSTOM FORENSIC HEAD ===
if hasattr(model, 'classifier'):
    # Determine input features
    if isinstance(model.classifier, nn.Linear):
        in_features = model.classifier.in_features
    elif isinstance(model.classifier, nn.Sequential):
        for layer in model.classifier:
            if isinstance(layer, nn.Linear):
                in_features = layer.in_features
                break
        
    # Replace classifier
    model.classifier = ForensicClassifier(in_features, num_classes=len(classes))
    print(f"✅ Custom Forensic Head with Attention Attached!")
else:
    print("⚠️ WARNING: Could not find 'classifier' layer to replace.")

# Move to GPU
model.to(DEVICE)

print(f"✅ Model Setup Complete: {MODEL_ID}")
print(f"   Input size: {INPUT_RESOLUTION}x{INPUT_RESOLUTION}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")


🤖 Loading model: Ateeqq/ai-vs-human-image-detector...

🔧 Checking model position embeddings for 512x512...
   Resizing embeddings: 196 patches -> 1024 patches
✅ Successfully resized position embeddings to 512x512
✅ Custom Forensic Head with Attention Attached!
✅ Model Setup Complete: Ateeqq/ai-vs-human-image-detector
   Input size: 512x512
   Parameters: 95,362,050


In [41]:
# ==========================================
# 6. TRAINING ARGUMENTS - MULTI-GPU OPTIMIZED
# ==========================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=MAX_EPOCHS,
    
    # Multi-GPU settings
    per_device_train_batch_size=BATCH_SIZE_PER_GPU,
    per_device_eval_batch_size=BATCH_SIZE_PER_GPU,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    
    # Optimization
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    
    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=25,
    
    # Performance optimizations
    fp16=True,  # Mixed precision
    dataloader_num_workers=4,  # More workers for 2 GPUs
    dataloader_pin_memory=True,
    gradient_checkpointing=False,  # We have enough memory
    
    # Multi-GPU strategy
    # Option 1: DataParallel (simpler, slightly slower)
    # Option 2: DistributedDataParallel (faster, more complex)
    # Transformers Trainer handles this automatically
    
    remove_unused_columns=False,
    push_to_hub=False,
    report_to=["tensorboard"],
    
    # For Kaggle
    disable_tqdm=False,
)


In [42]:
# ==========================================
# 7. TRAIN
# ==========================================

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    class_weights=class_weights,
)

print("\n" + "="*70)
print("🚀 STARTING TRAINING ON 2x T4 @ 512x512")
print("="*70)
print(f"Watch progress in TensorBoard: {OUTPUT_DIR}/logs")
print("="*70 + "\n")

train_result = trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)


🚀 STARTING TRAINING ON 2x T4 @ 512x512
Expected training time: ~4-5 hours
Watch progress in TensorBoard: /kaggle/working/zero_ai_forensic_512/logs



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Ai Accuracy,Real Accuracy,Fpr,Fnr
1,35.546500,3.124226,0.528899,0.611365,0.638472,0.586466,0.430108,0.586466,0.569892,0.413534
2,23.945500,1.304878,0.819477,0.852140,0.883065,0.823308,0.812903,0.823308,0.187097,0.176692
3,16.882500,2.263782,0.773555,0.785285,0.979401,0.655388,0.976344,0.655388,0.023656,0.344612
4,13.066900,1.212201,0.920032,0.936993,0.932919,0.941103,0.883871,0.941103,0.116129,0.058897
5,11.936700,1.803127,0.892320,0.907984,0.986765,0.840852,0.980645,0.840852,0.019355,0.159148
6,13.638700,1.547634,0.916865,0.930784,0.981919,0.884712,0.972043,0.884712,0.027957,0.115288
7,11.487600,1.346095,0.938242,0.950000,0.972441,0.928571,0.954839,0.928571,0.045161,0.071429
8,11.589200,1.568390,0.920032,0.933421,0.984701,0.887218,0.976344,0.887218,0.023656,0.112782
9,12.707700,1.667336,0.926366,0.938936,0.986207,0.895990,0.978495,0.895990,0.021505,0.104010


KeyboardInterrupt: 

Exception in thread Thread-7634 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resourc

In [46]:
# ==========================================
# 8. FINAL EVALUATION
# ==========================================
print("\n📊 Final Evaluation on Test Set...")
eval_results = trainer.evaluate()

print("\n" + "="*70)
print("📈 FINAL RESULTS @ 512x512 RESOLUTION")
print("="*70)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key:20s}: {value:.4f}")
print("="*70)


📊 Final Evaluation on Test Set...

📈 FINAL RESULTS @ 512x512 RESOLUTION
eval_loss           : 1.6673
eval_accuracy       : 0.9264
eval_f1             : 0.9389
eval_precision      : 0.9862
eval_recall         : 0.8960
eval_ai_accuracy    : 0.9785
eval_real_accuracy  : 0.8960
eval_fpr            : 0.0215
eval_fnr            : 0.1040


In [47]:
# ==========================================
# 9. SAVE MODEL
# ==========================================
print(f"\n💾 Saving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Save config for reference
config_info = {
    "resolution": INPUT_RESOLUTION,
    "num_gpus": NUM_GPUS,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
    "effective_batch_size": BATCH_SIZE_PER_GPU * NUM_GPUS * GRAD_ACCUM_STEPS,
    "model": MODEL_ID,
    "final_metrics": {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                      for k, v in eval_results.items()}
}

import json
with open(f"{OUTPUT_DIR}/training_config.json", "w") as f:
    json.dump(config_info, f, indent=2)

print("\n✅ ALL DONE!")
print(f"Model saved at: {OUTPUT_DIR}")
print(f"Config saved at: {OUTPUT_DIR}/training_config.json")
print("\n" + "="*70)
print("🎯 Next Steps:")
print("   1. Test with TTA inference (tta_inference.py)")
print("   2. Analyze failure cases")
print("   3. Deploy to production")
print("="*70)


💾 Saving model to /kaggle/working/zero_ai_forensic_512...

✅ ALL DONE!
Model saved at: /kaggle/working/zero_ai_forensic_512
Config saved at: /kaggle/working/zero_ai_forensic_512/training_config.json

🎯 Next Steps:
   1. Test with TTA inference (tta_inference.py)
   2. Analyze failure cases
   3. Deploy to production


In [49]:
import shutil
import os
from IPython.display import FileLink

# Define paths
folder_to_zip = "/kaggle/working/zero_ai_forensic_512"
output_filename = "zero_ai_forensic_512_model"

print(f"📦 Zipping folder '{folder_to_zip}'...")

# Create zip file
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"✅ Zip created: {output_filename}.zip")
print(f"   Size: {os.path.getsize(output_filename + '.zip') / (1024*1024):.2f} MB")

# Create a clickable download link (works in some Kaggle versions)
FileLink(f"{output_filename}.zip")

📦 Zipping folder '/kaggle/working/zero_ai_forensic_512'...
✅ Zip created: zero_ai_forensic_512_model.zip
   Size: 3222.73 MB


/kaggle/working/zero_ai_forensic_512_model.zip